# Hypothesis Testing: Risk Drivers Analysis

Test four key hypotheses about risk drivers:
1. No risk differences across provinces
2. No risk differences between zip codes
3. No margin (profit) difference between zip codes
4. No risk difference between Women and Men

**Metrics:**
- Claim Frequency: proportion of policies with at least one claim
- Claim Severity: average claim amount, given a claim occurred
- Margin: TotalPremium - TotalClaims

In [ ]:
import numpy as np
import pandas as pd
import os
import sys
sys.path.append(os.path.abspath(".."))
from src.data_loader import load_data
from src.hypothesis_tests import test_hypothesis

In [ ]:
# Load cleaned data
df = load_data("../data/MachineLearningRating_cleaned.csv")

# Create derived metrics
df["Loss_Ratio"] = df["TotalClaims"] / df["TotalPremium"].replace(0, np.nan)
df["Margin"] = df["TotalPremium"] - df["TotalClaims"]

print(f"Dataset shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nFirst few rows:")
print(df.head())

## Hypothesis 1: Risk Differences Across Provinces

**H₀:** There are no risk differences across provinces.

**KPI:** Claim Frequency

**Test:** Chi-squared test (categorical data)
- Group A (Control): Western Cape
- Group B (Test): All other provinces

In [ ]:
# Check available provinces
print("Province distribution:")
print(df["Province"].value_counts())

In [ ]:
# Hypothesis 1: Risk differences across provinces
h1_result = test_hypothesis(df, "Province", "Western Cape", kpi_type="frequency")

print("Hypothesis 1: Risk Differences Across Provinces")
print(f"Test: {h1_result.get('test')}")
print(f"Group A (Western Cape) size: {h1_result.get('group_a_size')}")
print(f"Group A (Western Cape) claim frequency: {h1_result.get('group_a_freq'):.4f}")
print(f"Group B (Other Provinces) size: {h1_result.get('group_b_size')}")
print(f"Group B (Other Provinces) claim frequency: {h1_result.get('group_b_freq'):.4f}")
print(f"p-value: {h1_result.get('p_value'):.6f}")

h1_rejected = h1_result.get('p_value', 1) < 0.05
h1_decision = "REJECT H₀" if h1_rejected else "FAIL TO REJECT H₀"
print(f"Decision: {h1_decision}")

## Hypothesis 2: Risk Differences Between Zip Codes

**H₀:** There are no risk differences between zip codes.

**KPI:** Claim Frequency

**Test:** Chi-squared test
- Group A (Control): Zip codes with highest volume
- Group B (Test): Zip codes with lower volume

In [ ]:
# Check zip codes with sufficient volume
print("Top zip codes by volume:")
top_zips = df["PostalCode"].value_counts().head(10)
print(top_zips)

# Use the most common zip code
top_zip = top_zips.index[0]
print(f"\nUsing top zip code: {top_zip}")

In [ ]:
# Hypothesis 2: Risk differences between zip codes
h2_result = test_hypothesis(df, "PostalCode", top_zip, kpi_type="frequency")

print("Hypothesis 2: Risk Differences Between Zip Codes")
print(f"Test: {h2_result.get('test')}")
print(f"Group A (Zip {top_zip}) size: {h2_result.get('group_a_size')}")
print(f"Group A (Zip {top_zip}) claim frequency: {h2_result.get('group_a_freq'):.4f}")
print(f"Group B (Other Zips) size: {h2_result.get('group_b_size')}")
print(f"Group B (Other Zips) claim frequency: {h2_result.get('group_b_freq'):.4f}")
print(f"p-value: {h2_result.get('p_value'):.6f}")

h2_rejected = h2_result.get('p_value', 1) < 0.05
h2_decision = "REJECT H₀" if h2_rejected else "FAIL TO REJECT H₀"
print(f"Decision: {h2_decision}")

## Hypothesis 3: Margin Difference Between Zip Codes

**H₀:** There is no significant margin (profit) difference between zip codes.

**KPI:** Margin (TotalPremium - TotalClaims)

**Test:** Independent t-test (continuous data)

In [ ]:
# Hypothesis 3: Margin difference between zip codes
h3_result = test_hypothesis(df, "PostalCode", top_zip, kpi_type="margin")

print("Hypothesis 3: Margin Difference Between Zip Codes")
print(f"Test: {h3_result.get('test')}")
print(f"Group A (Zip {top_zip}) size: {h3_result.get('group_a_size')}")
print(f"Group A (Zip {top_zip}) mean margin: R{h3_result.get('group_a_margin'):.2f}")
print(f"Group B (Other Zips) size: {h3_result.get('group_b_size')}")
print(f"Group B (Other Zips) mean margin: R{h3_result.get('group_b_margin'):.2f}")
print(f"p-value: {h3_result.get('p_value'):.6f}")

h3_rejected = h3_result.get('p_value', 1) < 0.05
h3_decision = "REJECT H₀" if h3_rejected else "FAIL TO REJECT H₀"
print(f"Decision: {h3_decision}")

## Hypothesis 4: Risk Difference Between Women and Men

**H₀:** There is no significant risk difference between Women and Men.

**KPI:** Claim Frequency

**Test:** Chi-squared test

In [ ]:
# Check gender distribution
print("Gender distribution:")
print(df["Gender"].value_counts())
print(f"\nMissing gender values: {df['Gender'].isna().sum()}")

In [ ]:
# Hypothesis 4: Risk difference between genders
df_gender = df[df["Gender"].notna()].copy()  # Remove missing values

# Determine which gender to use as control (typically the larger group)
gender_counts = df_gender["Gender"].value_counts()
control_gender = gender_counts.index[0]

h4_result = test_hypothesis(df_gender, "Gender", control_gender, kpi_type="frequency")

print("Hypothesis 4: Risk Difference Between Women and Men")
print(f"Test: {h4_result.get('test')}")
print(f"Group A ({control_gender}) size: {h4_result.get('group_a_size')}")
print(f"Group A ({control_gender}) claim frequency: {h4_result.get('group_a_freq'):.4f}")
print(f"Group B (Other Gender) size: {h4_result.get('group_b_size')}")
print(f"Group B (Other Gender) claim frequency: {h4_result.get('group_b_freq'):.4f}")
print(f"p-value: {h4_result.get('p_value'):.6f}")

h4_rejected = h4_result.get('p_value', 1) < 0.05
h4_decision = "REJECT H₀" if h4_rejected else "FAIL TO REJECT H₀"
print(f"Decision: {h4_decision}")

## Results Summary Table

In [ ]:
# Create results table
results_data = {
    "Hypothesis": [
        "H₀: No risk differences across provinces",
        "H₀: No risk differences between zip codes",
        "H₀: No margin difference between zip codes",
        "H₀: No risk difference between genders"
    ],
    "KPI": ["Claim Frequency", "Claim Frequency", "Margin", "Claim Frequency"],
    "Test": [
        h1_result.get('test'),
        h2_result.get('test'),
        h3_result.get('test'),
        h4_result.get('test')
    ],
    "p-value": [
        f"{h1_result.get('p_value'):.6f}",
        f"{h2_result.get('p_value'):.6f}",
        f"{h3_result.get('p_value'):.6f}",
        f"{h4_result.get('p_value'):.6f}"
    ],
    "Decision (α=0.05)": [
        "REJECT H₀" if h1_rejected else "FAIL TO REJECT",
        "REJECT H₀" if h2_rejected else "FAIL TO REJECT",
        "REJECT H₀" if h3_rejected else "FAIL TO REJECT",
        "REJECT H₀" if h4_rejected else "FAIL TO REJECT"
    ]
}

results_df = pd.DataFrame(results_data)
print("\n" + "="*100)
print("HYPOTHESIS TESTING RESULTS")
print("="*100)
print(results_df.to_string(index=False))
print("="*100)

## Business Recommendations

In [ ]:
recommendations = []

print("\n" + "="*100)
print("BUSINESS RECOMMENDATIONS")
print("="*100)

if h1_rejected:
    rec = f"\n1. PROVINCE-BASED RISK ADJUSTMENT (p={float(h1_result.get('p_value')):.4f})\n"
    rec += f"   Western Cape claim frequency: {h1_result.get('group_a_freq'):.2%}\n"
    rec += f"   Other provinces claim frequency: {h1_result.get('group_b_freq'):.2%}\n"
    diff_pct = (h1_result.get('group_b_freq') - h1_result.get('group_a_freq')) / h1_result.get('group_a_freq') * 100
    rec += f"   Risk difference: {diff_pct:+.1f}%\n"
    rec += "   RECOMMENDATION: Implement province-based premium adjustments. Provinces with higher claim\n"
    rec += "   frequencies warrant higher premiums to reflect increased risk."
    print(rec)
    recommendations.append(rec)
else:
    print("\n1. PROVINCE-BASED PRICING:")
    print("   No significant evidence of risk differences across provinces (p >= 0.05).")
    print("   RECOMMENDATION: Province-based pricing adjustments are not warranted at this time.")

if h2_rejected:
    rec = f"\n2. ZIP CODE-BASED RISK SEGMENTATION (p={float(h2_result.get('p_value')):.4f})\n"
    rec += f"   Zip {top_zip} claim frequency: {h2_result.get('group_a_freq'):.2%}\n"
    rec += f"   Other zip codes claim frequency: {h2_result.get('group_b_freq'):.2%}\n"
    diff_pct = (h2_result.get('group_b_freq') - h2_result.get('group_a_freq')) / h2_result.get('group_a_freq') * 100
    rec += f"   Risk difference: {diff_pct:+.1f}%\n"
    rec += "   RECOMMENDATION: Develop zip code-based segmentation for targeting low-risk areas\n"
    rec += "   with competitive premiums to attract new customers."
    print(rec)
    recommendations.append(rec)
else:
    print("\n2. ZIP CODE-BASED TARGETING:")
    print("   No significant evidence of risk differences between zip codes (p >= 0.05).")
    print("   RECOMMENDATION: Geographic targeting at zip code level is not justified by data.")

if h3_rejected:
    rec = f"\n3. ZIP CODE PROFITABILITY FOCUS (p={float(h3_result.get('p_value')):.4f})\n"
    rec += f"   Zip {top_zip} mean margin: R{h3_result.get('group_a_margin'):.2f}\n"
    rec += f"   Other zip codes mean margin: R{h3_result.get('group_b_margin'):.2f}\n"
    margin_diff = h3_result.get('group_b_margin') - h3_result.get('group_a_margin')
    rec += f"   Margin difference: R{margin_diff:+.2f}\n"
    rec += "   RECOMMENDATION: Prioritize marketing and customer acquisition in higher-margin zip codes\n"
    rec += "   to maximize portfolio profitability."
    print(rec)
    recommendations.append(rec)
else:
    print("\n3. ZIP CODE PROFITABILITY:")
    print("   No significant evidence of margin differences between zip codes (p >= 0.05).")
    print("   RECOMMENDATION: Profit margins are consistent across zip codes.")

if h4_rejected:
    rec = f"\n4. GENDER-BASED PRICING STRATEGY (p={float(h4_result.get('p_value')):.4f})\n"
    rec += f"   {control_gender} claim frequency: {h4_result.get('group_a_freq'):.2%}\n"
    rec += f"   Other gender claim frequency: {h4_result.get('group_b_freq'):.2%}\n"
    diff_pct = (h4_result.get('group_b_freq') - h4_result.get('group_a_freq')) / h4_result.get('group_a_freq') * 100
    rec += f"   Risk difference: {diff_pct:+.1f}%\n"
    rec += "   RECOMMENDATION: Implement gender-based premium adjustments to reflect actuarial evidence\n"
    rec += "   of risk differences. Ensure compliance with regulatory requirements."
    print(rec)
    recommendations.append(rec)
else:
    print("\n4. GENDER-BASED PRICING:")
    print("   No significant evidence of risk differences between genders (p >= 0.05).")
    print("   RECOMMENDATION: Gender-based pricing adjustments are not statistically warranted.")

print("\n" + "="*100)